# GPU stage - batch transformer embeddings (T4 / A100)
Takes `embed_input/` (produced by the Spark ETL, ~5 KB per opinion) and writes `embeddings/` (384 floats per opinion).
**Runtime -> Change runtime type -> GPU (T4).** Safe to re-run after a disconnect: finished shards are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Put the repo folder and data/exchange/embed_input in Drive (or upload them). Adjust:
REPO = '/content/drive/MyDrive/digital-archaeology'
EX   = REPO + '/data/exchange'

In [ ]:
%cd {REPO}
!pip -q install -r requirements-gpu.txt
import torch; print(torch.cuda.get_device_name(0))

In [ ]:
import sys; sys.path.insert(0, REPO + '/src')
from da.embed import main
stats = main(['--input', EX + '/embed_input', '--output', EX + '/embeddings',
              '--model', 'sentence-transformers/all-MiniLM-L6-v2', '--max-len', '256', '--batch-size', '256',
              '--scale', '1.0'])

In [ ]:
# Optional: throughput sweep for the paper (batch size vs segments/s)
import shutil, json
res = {}
for bs in (64, 128, 256, 512):
    shutil.rmtree(EX + '/emb_tmp', ignore_errors=True)
    s = main(['--input', EX + '/embed_input', '--output', EX + '/emb_tmp', '--batch-size', str(bs), '--limit', '20000'])
    res[bs] = round(s['segments_per_s'])
print(res)   # copy into your results table

Then copy nothing else: `embeddings/` is already inside `data/exchange`. Back on the cluster run `cluster,marts`.